## EE 242 Lab 4 - Digital Filtering - Various Filters

**Hanlin Ma, Sparsh Dadhich, Amanda Zhang, Team AF07**

> **Curation note:** This notebook was reconstructed from an archived rendered notebook/PDF view because no local `.ipynb` file was available. A few long lines may remain clipped or compacted from that source.


In [ ]:
# We'll refer to this as the "import cell." Every module you import should be im[ported here]%matplotlib inlineimport numpy as npimport matplotlibimport scipy.signal as sigimport matplotlib.pyplot as plt%matplotlib inline%config InlineBackend.figure_format = 'retina'# import whatever other modules you use in this lab -- there are more that you n[eed]from scipy.io.wavfile import read, writefrom IPython.display import Audioimport IPython.display as ipdimport scipy.io.wavfile as wavfrom scipy.fftpack import fftfrom scipy.signal import freqz, firwin, butter, lfilter, dlti, convolve

## Briefing (summarized)This lab considers different types of digital filters (discrete-time, linear, time-invariant filters) and their characterization in time and frequency: low-pass, high-pass, band-pass and band-reject filters; the general LCCDE sum(a_k y[n-k]) = sum(b_m x[n-m]); FIR vs. IIR filters; and scipy.signal helper functions (`signal.lfilter(b, a, x)`, `signal.firwin(order, W[, pass_zero])`, `signal.butter(order, W, type)`, `signal.freqz(b, a)`).## SummaryIn this lab, we will consider different types of digital filters and look at their characterization in both time and frequency. Specifically, we'll revisit the lab 2 problem of removing noise from signals (or smoothing signals), then explore filter design methods. This is a 2-week lab.

## Assignment 1 -- Different Filter ImplementationsIn this lab, we will be using standard tools to design filters, and we'll want to view them in both the time and frequency domain. In this assignment, you will write and test functions for plotting the frequency response and the impulse response of a system given the filter coefficients {a, b}. This assignment will have three parts, A-C.**A.** The response that is most often illustrated is the magnitude frequency response on a dB scale. Write a function that takes as input the filter coefficients, an optional flag for plotting both the magnitude and phase of the frequency response, and an optional sampling frequency. The function should generate either a plot of the magnitude or both the magnitude and the phase side-by-side, depending on the flag, with the default being magnitude only. The magnitude of the frequency response should be plotted on a dB scale with a range of [-100,0]. If no sampling frequency is provided, use radians for the frequency axis; otherwise use a Hz scale.**B.** Write a second function that takes as input the filter coefficients and a desired impulse response length, computes and returns the impulse response, and also plots the impulse response using a stem plot.**C.** Test the functions by plotting the magnitude, phase and impulse responses of two lowpass filters with a frequency cut-off of 0.15. One should be an FIR filter designed using the signal.firwin function (order 20) and the other should be an IIR filter with the signal.butter function (order 10).

In [ ]:
# Assignment 1 - Different Filter Implementations# Part Adef plot_mag_freq_response(b, a, plot_phase, fs):  # plot_phase and fs are 0 by d[efault]    # First generate H(omega) as described above in Briefing section (or use fun[ction])    # Use signal.freqz to get your frequency response    w, h = sig.freqz(b, a, worN=512)  # from the intros    H_omega = 20 * np.log10(np.abs(h))  # from the intros    # If fs is 0, the x axis would be in radians, otherwise it would be in Hz ba[sed on fs]    if fs == 0:        x_values = w        x_label = 'Frequency (radians/sample)'    else:        # Convert to Hz        x_values = w * fs / (2 * np.pi)  # from the intros        x_label = 'Frequency (Hz)'    # Plot magnitude response    plt.figure(figsize=(10, 4))    plt.plot(x_values, H_omega, label='Magnitude Response in dB')    plt.title('Magnitude Response')    plt.xlabel(x_label)    plt.ylabel('Magnitude (dB)')    plt.ylim([-100, 0])    plt.grid()    # If plot_phase is 0, do not plot the phase response, else plot the phase re[sponse]    if plot_phase != 0:        phase_angle = np.unwrap(np.angle(h))  # from the intros        plt.figure(figsize=(10, 4))        plt.plot(x_values, phase_angle, label='Phase Response in radians')        plt.title('Phase Response')        plt.xlabel(x_label)        plt.ylabel('Phase (radians)')        # plt.ylim([-90, 90])        plt.grid()    return# Part Bdef plot_impulse_response(b, a, impulse_length):    # Step 1: Use signal.lfilter to generate a filter    impulse_response = np.zeros(impulse_length)    impulse_response[0] = 1  # classical impulse    impulse_response_filtered = sig.lfilter(b, a, impulse_response)  # from intro    # Step 2: Plot impulse response    plt.figure(figsize=(10, 4))    plt.stem(np.arange(impulse_length), impulse_response_filtered, basefmt=" ")    plt.xlabel("n (samples)")    plt.ylabel("Amplitude")    plt.title("Impulse Response after filtered")    plt.grid()    # Step 3: return H(omega)    H_omega = impulse_response_filtered    return H_omega# Part C# Use signal.firwin and signal.butter to generate your b and a coefficients, the[n test]#Lowpass: b = signal.firwin(order, W)#Highpass: b = signal.firwin(order, W, pass_zero = False)# FIR filter designed using the signal.firwin function (order 20)cut_off = 0.15FIR_order = 21  # 0-20FIR_filter_coefficient_a = [1.0]FIR_filter_coefficient_b = sig.firwin(FIR_order, cut_off)  # order 20(0-20) so 21plot_mag_freq_response(FIR_filter_coefficient_b, FIR_filter_coefficient_a, plot_[phase=1, fs=0])FIR_impulse = plot_impulse_response(FIR_filter_coefficient_b, FIR_filter_coeffic[ient_a, 50])# IIR filter with the signal.butter function (order 10), use  b, a = signal.butt[er(...)]IIR_order = 10IIR_filter_coefficient_b, IIR_filter_coefficient_a = sig.butter(IIR_order, cut_o[ff])plot_mag_freq_response(IIR_filter_coefficient_b, IIR_filter_coefficient_a, plot_[phase=1, fs=0])IIR_impulse = plot_impulse_response(IIR_filter_coefficient_b, IIR_filter_coeffic[ient_a, 50])plt.show()

### Discussion (Assignment 1, transcribed)The FIR filter (firwin) requires a higher order to achieve a similar roll-off compared to the IIR filter; the IIR filter (using butter) achieves a sharper roll-off with a lower order (10), making it more efficient in terms of computational cost.Phase Response: The FIR filter exhibits a linear phase shape. The IIR filter has nonlinear phase, which can introduce signal distortion.Impulse Response: The FIR filter has a finite impulse response (plot_impulse_response shows it eventually reaches or converge to zero). The IIR filter has an infinite impulse response, meaning it continues indefinitely (diverge), which may introduce stability concerns.Trade-offs: FIR filters are computationally expensive but ideal for applications requiring phase linearity (audio processing). IIR filters are more efficient but can distort phase and may be less stable.

## Assignment 2 -- Different Filter Implementations for Smoothing SignalsIn lab 2B, you experimented with smoothing a noisy signal using a moving average window and a convolution. The convolution used an impulse response h[n] that was a causal version of the moving average window. In this problem, you will implement the smoothing function using the both convolution and the signal.lfilter command, to see that they give the same result. This assignment will have three parts, A-C.**A.** Using the code from lab 2, create a base time signal and a noisy version of it by adding random noise generated with the numpy.random.randn() function (the standard normal distribution, which is zero mean and unit variance). Plot the original and noisy signals together with the original overlaid on the noisy version, with the time axis labeled assuming a sampling rate of 1000 Hz. Constrain the y-axis to be [0,25] for all plots. Include a legend with the plot.**B.** Create one smoothed version of the signal called filtsig1 by using the convolve function from lab 2B with the box impulse response and k=10. Create a second version called filtsig2 by using the signal.lfilter function. Recall that for the FIR filter, the impulse response is equal to the b coefficient vector. Plot the two filtered signals overlaid. Recall that the convolve function will change the length, so you will need to define a new time vector for that. You should find that the two methods give the same result except for edge effects.**C.** Use the function that you wrote in assignment 1 to plot the magnitude and phase for the frequency response of this filter. It should look like a low pass filter.

In [ ]:
# Assignment 2 - Different Filter Implementations for Smoothing Signals# set up relevant parameterssrate = 1000  # sampling rate in Hztime = np.arange(0, 2, 1 / srate)  # associated time vector that corresponds to 2 se[conds]n = len(time)  # length of the time vector# here is a base signal to work with, values of signal points chosen randomlyp = 10  # points for piecewise linear signalamp = 20  # amplitude range of base signalbase = np.interp(np.linspace(0, p, n), np.arange(0, p), np.random.rand(p) * amp)# create some random noise to be added to the abve base signalsnoiseamp = 2noise = noiseamp * np.random.randn(n)# Part A# Create a noisy signal and overlapping plot with base and noisy signal (use a l[egend])noisy_signal = base + noise# Plot signalsplt.figure(figsize=(10, 4))plt.plot(time, base, label="Original Signal", linewidth=2, color='red')plt.plot(time, noisy_signal, label="Noisy Signal", alpha=0.7, color='blue')plt.title("Original and Noisy Signals")plt.xlabel("Time (seconds)")plt.ylabel("Amplitude")plt.ylim([0, 25])plt.legend()plt.grid()plt.show()# Part B# Use convolution from lab 2B and signal.lfilter to apply your filter# code from lab 2bk = 20N = 2 * k + 1hfilt = np.ones(N) / Nconvolve_filter = np.convolve(noisy_signal, hfilt, mode='same')convolution_axis = np.arange(0, len(convolve_filter)) / sratelfilter_filter = sig.lfilter(hfilt, 1, noisy_signal)plt.figure(figsize=(10, 5))plt.plot(convolution_axis, convolve_filter, label="Filtered Signal by Convolutio[n]")plt.plot(time, lfilter_filter, label="Filtered Signal (lfilter)", linestyle='--[ ]')plt.title("Comparison between convolution and lfilter")plt.xlabel("Time (seconds)")plt.ylabel("Amplitude")plt.ylim([0, 25])plt.legend()plt.grid()plt.show()# Part C# Use function to plot magnitude and phasedef plot_mag_freq_response(b, a, plot_phase, fs):  # plot_phase and fs are 0 by d[efault]    w, h = sig.freqz(b, a, worN=512)    H_omega = 20 * np.log10(np.abs(h))    if fs == 0:        x_values = w        x_label = 'Frequency (radians/sample)'    else:        x_values = w * fs / (2 * np.pi)        x_label = 'Frequency (Hz)'    plt.figure(figsize=(10, 4))    plt.plot(x_values, H_omega, label='Magnitude Response in dB')    plt.title('Magnitude Response')    plt.xlabel(x_label)    plt.ylabel('Magnitude (dB)')    plt.ylim([-100, 0])    plt.grid()    if plot_phase != 0:        phase_angle = np.unwrap(np.angle(h))        plt.figure(figsize=(10, 4))        plt.plot(x_values, phase_angle, label='Phase Response in radians')        plt.title('Phase Response')        plt.xlabel(x_label)        plt.ylabel('Phase (radians)')        plt.grid()    returnplot_mag_freq_response(hfilt, [1.0], plot_phase=1, fs=0)

### Discussion (Assignment 2, transcribed)The moving window average (and its causal version) is an FIR filter, so the phase should be linear. How might the result change if you used a Butterworth filter?A Butterworth filter is an IIR filter, which introduces nonlinear phase distortion, unlike the moving average FIR filter, which has a linear phase response, like what it should in previews assignment.The magnitude response would be sharper.The phase response would be nonlinear, meaning different frequency components would experience different delays, potentially distorting the signal shape.The impulse response will tend to be diverge at the end (not converge to zero)A Butterworth filter would provide better frequency selectivity but at the cost of phase distortion and potential signal shape alterations.